# 22. 🧰 Прикладное: байесовские методы и неопределённость

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/justxor/math-for-ai-2026/blob/main/notebooks/22_bayes_uncertainty.ipynb)

Код из раздела [README.md](https://github.com/justxor/math-for-ai-2026/blob/main/README.md#m22). Запускайте ячейки сверху вниз: последующие используют переменные предыдущих. Теория, формулы и картинки — в тексте курса по ссылке.

In [ ]:
# Colab: всё нужное уже установлено. Локально: pip install -r requirements.txt
import numpy as np
import math
np.set_printoptions(precision=4, suppress=True)

### 🏋️ Практика модуля

**22.1.** Байесовский A/B: A — 120 из 2400, B — 150 из 2400. Найдите $P(p_B > p_A)$.

*▶️ Решение — сначала попробуйте сами.*

In [ ]:
import numpy as np
rng = np.random.default_rng(0)
pa = rng.beta(1 + 120, 1 + 2280, 200_000); pb = rng.beta(1 + 150, 1 + 2250, 200_000)
print((pb > pa).mean().round(3), np.percentile(pb - pa, [2.5, 97.5]).round(4))
# ≈ 0.97; 95% интервал разницы ≈ [−0.0006, 0.0256]

### 🏋️ Практика модуля

**22.2.** Обучите ансамбль из 5 полиномиальных моделей на бутстрап-выборках и покажите, что разброс растёт вне области данных.

*▶️ Решение — сначала попробуйте сами.*

In [ ]:
x = rng.uniform(-2, 2, 60); y = np.sin(x) + rng.normal(0, 0.1, 60)
xs = np.array([0.0, 1.5, 3.0, 4.0])                       # 3 и 4 — вне обучающих данных
preds = []
for _ in range(5):
    idx = rng.integers(0, 60, 60)
    preds.append(np.polyval(np.polyfit(x[idx], y[idx], 5), xs))
print(np.std(preds, axis=0).round(3))                    # маленький разброс внутри [−2, 2], большой снаружи

### 🏋️ Практика модуля

**22.3.** Реализуйте split conformal для регрессии и проверьте покрытие на новых данных.

*▶️ Решение — сначала попробуйте сами.*

In [ ]:
X = rng.uniform(0, 10, 2000); Y = np.sin(X) + rng.normal(0, 0.3, 2000)
tr, cal, te = slice(0, 1000), slice(1000, 1500), slice(1500, 2000)
coef = np.polyfit(X[tr], Y[tr], 7)
s = np.abs(Y[cal] - np.polyval(coef, X[cal])); n = s.size
q = np.quantile(s, np.ceil((n + 1) * 0.9) / n)
cover = np.mean(np.abs(Y[te] - np.polyval(coef, X[te])) <= q)
print(round(q, 3), round(cover, 3))                     # покрытие ≈ 0.88–0.92: гарантия «в среднем», конкретная выборка колеблется

## 🔬 Лаборатория: иллюстрации модуля

Это код картинок из курса. Меняйте параметры (learning rate, температуру, число точек, ядро…) и перезапускайте ячейку — так интуиция появляется быстрее всего.

In [ ]:
# Помощники для иллюстраций (те же, что в scripts/make_figures.py)
import matplotlib.pyplot as plt
from scipy import stats
C = ["#2563eb", "#dc2626", "#059669", "#d97706", "#7c3aed", "#0891b2"]
plt.rcParams.update({"figure.dpi": 100, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.alpha": 0.25})

def save(fig, name):          # в ноутбуке просто показываем картинку
    fig.tight_layout(); plt.show()

def arrow(ax, v, color, label=None, origin=(0, 0), **kw):
    ax.annotate("", xy=(origin[0] + v[0], origin[1] + v[1]), xytext=origin,
                arrowprops=dict(arrowstyle="-|>", color=color, lw=2.2, **kw))
    if label:
        ax.text(origin[0] + v[0] * 1.08, origin[1] + v[1] * 1.08, label, color=color, fontsize=13, fontweight="bold")

def rosen_like(x, y):
    return 0.5 * x**2 + 4 * y**2

In [ ]:
def bayes_conformal():
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    p = np.linspace(0, 1, 400)
    for (a, b, lab), c in zip([(1, 1, "априор Beta(1,1)"), (3, 9, "после 2 из 10"), (21, 81, "после 20 из 100"),
                               (201, 801, "после 200 из 1000")], C):
        axes[0].plot(p, stats.beta.pdf(p, a, b), color=c, lw=2, label=lab)
    axes[0].set_xlim(0, 0.5); axes[0].legend(fontsize=9)
    axes[0].set_title("Апостериор конверсии сужается с ростом данных")
    rng = np.random.default_rng(3)
    x = np.sort(rng.uniform(0, 10, 300)); y = np.sin(x) + rng.normal(0, 0.15 + 0.05 * x, 300)
    tr, cal = np.arange(0, 300, 2), np.arange(1, 300, 2)
    coef = np.polyfit(x[tr], y[tr], 5)
    resid = np.abs(y[cal] - np.polyval(coef, x[cal]))
    q = np.quantile(resid, np.ceil((len(cal) + 1) * 0.9) / len(cal))
    xs = np.linspace(0, 10, 300); f = np.polyval(coef, xs)
    axes[1].scatter(x, y, s=6, color="gray", alpha=0.6)
    axes[1].plot(xs, f, color=C[0], lw=2, label="модель")
    axes[1].fill_between(xs, f - q, f + q, color=C[0], alpha=0.2, label=f"90% conformal-интервал (±{q:.2f})")
    xt = rng.uniform(0, 10, 5000); yt = np.sin(xt) + rng.normal(0, 0.15 + 0.05 * xt, 5000)
    cover = np.mean(np.abs(yt - np.polyval(coef, xt)) <= q)
    axes[1].legend(fontsize=9); axes[1].set_title(f"Conformal prediction: покрытие на новых данных = {cover:.1%}")
    save(fig, "bayes_conformal")

bayes_conformal()